# Tema 8 · Bootstrapping y uso de datos no etiquetados

**Trabajo de investigación aplicada · Minería de Datos · grupal**

**Integrantes del grupo:** _escriba aquí los nombres_

## 📖 El caso

**Buses Itata Sur** hace el recorrido rural Quirihue–Ninhue–Chillán. Para su informe anual, la gerencia hizo una encuesta de satisfacción, pero solo **40 pasajeros** la contestaron, y quiere informar «el porcentaje de pasajeros satisfechos». Además, quiere un modelo que **prediga** si un viaje deja satisfecho al pasajero según el atraso, la limpieza, la ocupación y la duración. De otros **400 viajes** tiene esos datos pero no la encuesta. Una **auditoría externa** revisó 200 viajes al azar: esos datos sirven solo para comprobar los resultados.

> **❓ La pregunta del cliente:** ¿Cuánto se puede confiar en el porcentaje de satisfechos de 40 encuestas, y sirven de algo los 400 viajes sin encuesta?

## 🗂️ Los datos: `t8_buses.csv` (640 viajes: 40 con encuesta, 400 sin encuesta y 200 de la auditoría)

| Columna | Qué significa |
|---|---|
| `viaje` | número del viaje |
| `atraso_min` | minutos de atraso en la llegada |
| `limpieza_nota` | nota de limpieza del bus, de 1 a 7 |
| `ocupacion_pct` | porcentaje de asientos ocupados |
| `duracion_min` | duración del viaje, en minutos |
| `satisfecho` | **Sí / No** si el pasajero quedó satisfecho; **vacío** en los viajes sin encuesta |
| `conjunto` | `etiquetado` (con encuesta), `sin_etiqueta` (sin encuesta) o `evaluacion` (auditoría) |

## Cómo trabajar este notebook

1. **Archivo → Guardar una copia en Drive.** Trabajen siempre en esa copia. Para trabajar en grupo, el dueño de la copia la **comparte** con el resto (botón **Compartir**).
2. **Todo el código viene escrito y explicado.** No hay que programar desde cero: lean la explicación 📘 que está antes de cada celda y ejecútenla (▶) **en orden**.
3. Las líneas **`# ✅ Debe mostrar:`** dicen qué debe aparecer. Si aparece otra cosa, hagan **Entorno de ejecución → Reiniciar y ejecutar todo**.
4. En las celdas **✏️ Experimento** cambien el valor que se indica, vuelvan a ejecutar y anoten lo que pasa en la tabla de abajo. **Al terminar, dejen el valor original.**
5. Respondan cada **✍️** con **sus números**. Esas respuestas son la materia prima de la **parte práctica del informe** y de la **exposición**.

| Ícono | Qué es |
|:-:|---|
| 📘 | Explicación de lo que hace la celda siguiente |
| ✅ | Lo que debe mostrar la celda |
| ✏️ | Experimento: ustedes cambian un valor |
| ✍️ | Pregunta para responder con sus números |
| 🧠 | De la información al conocimiento: convertir el resultado en una decisión para el cliente |

In [ ]:
# ▶ CELDA DE INICIO — no la modifique. Importa las librerías y carga los datos.
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

RUTA = "https://raw.githubusercontent.com/de-mag-ia/diguillin-datos/main/investigacion/"
datos = pd.read_csv(RUTA + "t8_buses.csv")

print(f"Datos cargados: {len(datos)} filas y {datos.shape[1]} columnas.")
# ✅ Debe mostrar: Datos cargados: 640 filas y 7 columnas.

## 1. Conocer los datos

📘 Cada fila es un **viaje** del recorrido Quirihue–Ninhue–Chillán, con su **atraso** (minutos), la **nota de
limpieza** del bus (1 a 7), la **ocupación** (% de asientos usados) y la **duración** del viaje. La columna
`conjunto` dice de dónde viene cada viaje:

| `conjunto` | Qué es | ¿Tiene `satisfecho`? |
|---|---|:-:|
| `etiquetado` | Viajes donde un pasajero **contestó la encuesta** | Sí |
| `sin_etiqueta` | Viajes **sin encuesta**: se conocen los datos del viaje, pero no si el pasajero quedó satisfecho | **No** (celda vacía) |
| `evaluacion` | Viajes revisados por una **auditoría externa** con entrevistas a pasajeros al azar | Sí |

La celda separa los tres conjuntos. `isna()` cuenta las celdas vacías de `satisfecho`.

In [ ]:
datos.head()

In [ ]:
etiquetados = datos[datos["conjunto"] == "etiquetado"]
sin_etiqueta = datos[datos["conjunto"] == "sin_etiqueta"]
auditoria = datos[datos["conjunto"] == "evaluacion"]
print("Con encuesta:", len(etiquetados), "· Sin encuesta:", len(sin_etiqueta), "· Auditoría:", len(auditoria))
print("Celdas vacías en 'satisfecho':", datos["satisfecho"].isna().sum())
# ✅ Debe mostrar: Con encuesta: 40 · Sin encuesta: 400 · Auditoría: 200  /  Celdas vacías en 'satisfecho': 400

# Parte A · Bootstrap: ¿cuánto confiar en un porcentaje?

## 2. El porcentaje de satisfechos según las encuestas

📘 La gerencia quiere informar «el porcentaje de pasajeros satisfechos». Con las encuestas se calcula fácil: se
cuentan los «Sí» y se divide por el total.

In [ ]:
respuestas = etiquetados["satisfecho"].values
porcentaje = (respuestas == "Sí").mean()
print(f"Satisfechos según las {len(respuestas)} encuestas: {porcentaje:.1%}")
# ✅ Debe mostrar: Satisfechos según las 40 encuestas: 82.5%

## 3. El bootstrap: los papelitos en una bolsa

📘 Si se hubieran encuestado **otros** 40 viajes, el porcentaje habría salido distinto. ¿Cuánto distinto? El
**bootstrap** lo estima **sin hacer encuestas nuevas**:

1. Se escribe cada una de las 40 respuestas en un **papelito** y se echan a una bolsa.
2. Se saca un papelito, se anota y **se devuelve a la bolsa** (*con reposición*). Se repite 40 veces. Resultado: una
   **remuestra**, una «encuesta imaginaria» de 40 respuestas (algunas repetidas, otras ausentes).
3. Se calcula el % de satisfechos de esa remuestra.
4. Se repite todo **1.000 veces** y se miran los 1.000 porcentajes.

El **intervalo del 95 %** va desde el porcentaje que deja abajo al 2,5 % de las remuestras hasta el que deja
arriba al 2,5 % (los *percentiles* 2,5 y 97,5). `rng.choice(..., replace=True)` es «sacar el papelito y devolverlo».
La semilla (`2026`) hace que todos los grupos obtengan las mismas remuestras.

In [ ]:
def intervalo_bootstrap(respuestas, n_remuestras):
    """Intervalo del 95 % del porcentaje de 'Sí', con remuestras con reposición."""
    rng = np.random.default_rng(2026)
    porcentajes = []
    for _ in range(n_remuestras):
        remuestra = rng.choice(respuestas, size=len(respuestas), replace=True)
        porcentajes.append((remuestra == "Sí").mean())
    inferior, superior = np.percentile(porcentajes, [2.5, 97.5])
    return inferior, superior, porcentajes

inferior, superior, porcentajes = intervalo_bootstrap(respuestas, 1000)
plt.figure(figsize=(6, 3.5))
plt.hist(porcentajes, bins=20, color="#90caf9", edgecolor="white")
plt.axvline(inferior, color="red", ls="--"); plt.axvline(superior, color="red", ls="--")
plt.xlabel("% de satisfechos en cada remuestra"); plt.ylabel("Cantidad de remuestras")
plt.title("1.000 encuestas imaginarias")
plt.show()
print(f"Intervalo del 95 %: entre {inferior:.0%} y {superior:.0%}")
# ✅ Debe mostrar: Intervalo del 95 %: entre 70% y 92%

### ✍️ Expliquen con sus palabras qué significa el intervalo que obtuvieron. ¿Cuántos puntos de ancho tiene? ¿Podría la empresa afirmar que «el 82,5 % de los pasajeros está satisfecho»?

_Su respuesta:_

## 4. ¿Más remuestras dan un intervalo más angosto?

📘 Cambien el número de remuestras y observen qué pasa con el intervalo.

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
N_REMUESTRAS = 1000        # ✏️ pruebe con 100, 1000 y 5000 (al terminar, deje 1000)

inf_exp, sup_exp, _ = intervalo_bootstrap(respuestas, N_REMUESTRAS)
print(f"{N_REMUESTRAS} remuestras → entre {inf_exp:.0%} y {sup_exp:.0%} (ancho: {100 * (sup_exp - inf_exp):.0f} puntos)")

**✏️ Anoten aquí sus resultados:**

| Remuestras | Límite inferior | Límite superior | Ancho del intervalo |
|:-:|:-:|:-:|:-:|
| 100 | | | |
| 1000 | | | |
| 5000 | | | |

### ✍️ Según su tabla, ¿el intervalo se angosta al usar más remuestras? ¿Por qué? Entonces, ¿qué habría que hacer para tener un intervalo más angosto?

_Su respuesta:_

## 5. Comparar con la auditoría externa

📘 La auditoría entrevistó a pasajeros de **200 viajes elegidos al azar**. La celda calcula su porcentaje y su
intervalo bootstrap, con la misma función.

In [ ]:
respuestas_auditoria = auditoria["satisfecho"].values
inf_a, sup_a, _ = intervalo_bootstrap(respuestas_auditoria, 1000)
print(f"Auditoría: {(respuestas_auditoria == 'Sí').mean():.1%} satisfechos · intervalo del 95 %: entre {inf_a:.0%} y {sup_a:.0%}")
print(f"Encuestas: {porcentaje:.1%} satisfechos · intervalo del 95 %: entre {inferior:.0%} y {superior:.0%}")
# ✅ Debe mostrar: Auditoría: 52.5% satisfechos · intervalo del 95 %: entre 46% y 60%  /  Encuestas: 82.5% satisfechos · intervalo del 95 %: entre 70% y 92%

### ✍️ Comparen los dos intervalos. ¿Por qué el de la auditoría es más angosto? ¿Se tocan los dos intervalos? Si no se tocan, ¿qué está pasando con las encuestas? ¿El bootstrap podía detectarlo?

_Su respuesta:_

# Parte B · Usar los viajes sin encuesta

## 6. Un modelo entrenado solo con las 40 encuestas

📘 Ahora la empresa quiere **predecir** si un viaje deja satisfecho al pasajero según su atraso, limpieza,
ocupación y duración. Solo 40 viajes tienen la respuesta (la **etiqueta**). Se entrena una **regresión
logística** con esos 40 y se mide su acierto con la **auditoría**, que el modelo nunca ve.

In [ ]:
COLUMNAS = ["atraso_min", "limpieza_nota", "ocupacion_pct", "duracion_min"]
modelo_40 = LogisticRegression(max_iter=1000).fit(etiquetados[COLUMNAS], etiquetados["satisfecho"])
acierto_40 = modelo_40.score(auditoria[COLUMNAS], auditoria["satisfecho"])
print(f"Entrenado con 40 encuestas → acierto en la auditoría: {acierto_40:.1%}")
# ✅ Debe mostrar: Entrenado con 40 encuestas → acierto en la auditoría: 84.0%

## 7. Autoentrenamiento: el modelo se enseña a sí mismo

📘 Hay **400 viajes sin encuesta**. No se sabe si esos pasajeros quedaron satisfechos, pero sí se conocen los datos
del viaje. El **autoentrenamiento** (*self-training*) los aprovecha así:

1. Se entrena el modelo con los viajes que tienen etiqueta.
2. El modelo predice los viajes sin etiqueta y dice **qué tan seguro** está (`predict_proba`).
3. Los viajes donde está **muy seguro** (probabilidad ≥ `UMBRAL`) se agregan a los datos de entrenamiento **con la
   etiqueta que predijo el modelo** (una *pseudoetiqueta*).
4. Se vuelve a entrenar con más datos. Se repite 3 **rondas**.

Al final, se mide de nuevo el acierto con la auditoría.

In [ ]:
def autoentrenar(umbral, mostrar=True):
    X_actual = etiquetados[COLUMNAS]
    y_actual = etiquetados["satisfecho"]
    pendientes = sin_etiqueta[COLUMNAS]
    for ronda in range(1, 4):
        if len(pendientes) == 0:
            break
        modelo = LogisticRegression(max_iter=1000).fit(X_actual, y_actual)
        seguridad = modelo.predict_proba(pendientes).max(axis=1)      # qué tan seguro está, de 0,5 a 1
        nuevos = pendientes[seguridad >= umbral]                         # los viajes donde está muy seguro
        if len(nuevos) > 0:
            X_actual = pd.concat([X_actual, nuevos])
            y_actual = pd.concat([y_actual, pd.Series(modelo.predict(nuevos), index=nuevos.index)])
        pendientes = pendientes[seguridad < umbral]                      # quedan para la ronda siguiente
        final = LogisticRegression(max_iter=1000).fit(X_actual, y_actual)
        acierto = final.score(auditoria[COLUMNAS], auditoria["satisfecho"])
        if mostrar:
            print(f"Ronda {ronda}: +{len(nuevos)} viajes · total {len(X_actual)} · acierto en auditoría: {acierto:.1%}")
    return acierto, len(X_actual) - len(etiquetados)

acierto_auto, agregados = autoentrenar(0.9)
# ✅ Debe mostrar: Ronda 1: +236 viajes · total 276 · acierto en auditoría: 86.5%  /  Ronda 2: +65 viajes · total 341 · acierto en auditoría: 89.0%  /  Ronda 3: +16 viajes · total 357 · acierto en auditoría: 89.5%

### ✍️ ¿Cuánto mejoró el acierto en la auditoría respecto del modelo con 40 encuestas? ¿Cómo es posible mejorar usando viajes de los que **no se sabe** la respuesta?

_Su respuesta:_

## 8. ¿Qué tan seguro debe estar el modelo?

📘 El `UMBRAL` decide qué viajes se aceptan. Cambien su valor y anoten el resultado.

In [ ]:
# ✏️ EXPERIMENTO — cambie el valor indicado y vuelva a ejecutar
UMBRAL = 0.9        # ✏️ pruebe con 0.6, 0.8, 0.9 y 0.99 (al terminar, deje 0.9)

acierto_exp, agregados_exp = autoentrenar(UMBRAL, mostrar=False)
print(f"Umbral {UMBRAL}: {agregados_exp} viajes agregados · acierto en auditoría: {acierto_exp:.1%}")

**✏️ Anoten aquí sus resultados:**

| Umbral | Viajes agregados (3 rondas) | Acierto en la auditoría | ¿Mejoró respecto de las 40 encuestas? |
|:-:|:-:|:-:|:-:|
| 0.6 | | | |
| 0.8 | | | |
| 0.9 | | | |
| 0.99 | | | |

### ✍️ Según su tabla, ¿qué pasa con un umbral muy bajo (0,6) y con uno muy alto (0,99)? ¿Cuál recomendarían?

_Su respuesta:_

### ✍️ 🧠 **De la información al conocimiento.** Escriban la recomendación para Buses Itata Sur: ¿qué porcentaje de satisfechos deberían informar y cómo? ¿Conviene usar los viajes sin encuesta o pagar más encuestas? ¿Qué cambiarían en la forma de encuestar?

_Su respuesta:_

---
## ✔️ Para entregar

1. Revisen que **todas** las ✍️ tengan respuesta y que las tablas ✏️ estén completas.
2. **Entorno de ejecución → Reiniciar y ejecutar todo**: todo debe correr sin errores.
3. **Archivo → Descargar → Descargar .ipynb** y súbanlo al **Aula Virtual** como `Grupo_N_Tema8.ipynb`, junto con el informe en PDF.